# English TTS Fine-tuning (LJSpeech) 
This notebook prepares the **LJSpeech** dataset (English), demonstrates **inference** with a pretrained Coqui TTS model (no training needed), and provides a compact, **workable** set of instructions & example command to fine-tune a TTS model.

**What this notebook does:**
- Download and prepare LJSpeech (using `datasets`).
- Run a short synthesis demo using a pretrained Coqui TTS model (works CPU/GPU).
- Save dataset in the structure commonly used for TTS training.
- Provide a minimal example config & CLI command for fine-tuning with Coqui TTS.

In [2]:
# Install minimal required libraries
!pip install -q TTS datasets soundfile tqdm
print('Installed packages')

Installed packages


ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0,<3.9; 0.0.10.3 Requires-Python >=3.6.0,<3.9; 0.0.11 Requires-Python >=3.6.0,<3.9; 0.0.12 Requires-Python >=3.6.0,<3.9; 0.0.13.1 Requires-Python >=3.6.0,<3.9; 0.0.13.2 Requires-Python >=3.6.0,<3.9; 0.0.14.1 Requires-Python >=3.6.0,<3.9; 0.0.15 Requires-Python >=3.6.0,<3.9; 0.0.15.1 Requires-Python >=3.6.0,<3.9; 0.0.9 Requires-Python >=3.6.0,<3.9; 0.0.9.1 Requires-Python >=3.6.0,<3.9; 0.0.9.2 Requires-Python >=3.6.0,<3.9; 0.0.9a10 Requires-Python >=3.6.0,<3.9; 0.0.9a9 Requires-Python >=3.6.0,<3.9; 0.1.0 Requires-Python >=3.6.0,<3.10; 0.1.1 Requires-Python >=3.6.0,<3.10; 0.1.2 Requires-Python >=3.6.0,<3.10; 0.1.3 Requires-Python >=3.6.0,<3.10; 0.10.0 Requires-Python >=3.7.0,<3.11; 0.10.1 Requires-Python >=3.7.0,<3.11; 0.10.2 Requires-Python >=3.7.0,<3.11; 0.11.0 Requires-Python >=3.7.0,<3.11; 0.11.1 Requires-Python >=3.7.0,<3.11; 0.12.0 Requires-Python >=3.7.0,<3.11; 0.13.0 Requ

In [3]:
# Download LJSpeech (via Hugging Face `datasets`) and prepare files
from datasets import load_dataset
import os
from pathlib import Path

dataset = load_dataset('ljspeech', 'clean', split='train')
print('Loaded LJSpeech examples:', len(dataset))

out_dir = Path('LJSpeech_dataset')
wavs_dir = out_dir / 'wavs'
out_dir.mkdir(exist_ok=True)
wavs_dir.mkdir(exist_ok=True)

metadata_lines = []
for i, ex in enumerate(dataset):

    # ex has: 'audio' -> {'path', 'array', 'sampling_rate'} and 'text'
    wav_path = wavs_dir / f'ljs_audio_{i:05d}.wav'

    # save audio bytes
    audio = ex['audio']

    # datasets may provide path or array; use the provided array to write wav
    import soundfile as sf
    sf.write(str(wav_path), audio['array'], audio['sampling_rate'])

    # create metadata line: filename|transcript (Coqui TTS default separator is '|')
    transcript = ex['text'].replace('|', ' ').strip()
    metadata_lines.append(f"{wav_path.name}|{transcript}")
    if i and (i % 200 == 0):
        print(f'Prepared {i} files')

# write metadata.csv
with open(out_dir / 'metadata.csv', 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata_lines))

print('Saved dataset to', out_dir)

FileNotFoundError: Couldn't find a dataset script at c:\Users\ashish\Documents\Major_Project\TTS\ljspeech\ljspeech.py or any data file in the same directory. Couldn't find 'ljspeech' on the Hugging Face Hub either: FileNotFoundError: Dataset 'ljspeech' doesn't exist on the Hub

In [ ]:
# Quick inference demo using a pretrained Coqui TTS model
from TTS.api import TTS
import IPython.display as ipd

# choose an English pretrained model (uses LJSpeech style voice)
model_name = 'tts_models/en/ljspeech/tacotron2-DDC'
print('Loading model:', model_name)
tts = TTS(model_name, progress_bar=True, gpu=False)  # gpu=True

sample_text = 'Hello! This is a short test using a pretrained English TTS model.'
out_wav = 'demo_output.wav'
tts.tts_to_file(text=sample_text, file_path=out_wav)
print('Saved demo audio to', out_wav)
ipd.Audio(out_wav)